# Correlation & Spike Analysis — Interactive

Interactive analysis of windowed correlation, spike detection, and case/control
attribution. This notebook provides visualization capabilities for the results
produced by `windowed_correlation.py`, `spike_detection.py`, and
`case_control_analysis.py`.

**Blueprint Reference:** §06.2 Windowed Correlation, §06.3 Spike Detection, §06.4 Case/Control

In [ ]:
import csv, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

DATA_DIR = Path('../data')
STATS_DIR = Path('../stats')
print('Available stats:', [f.name for f in STATS_DIR.glob('*.csv')])

## 1. Load Windowed Correlation Data

In [ ]:
def load_csv(name):
    path = STATS_DIR / name
    if not path.exists():
        print(f'{name} not found — run the corresponding analysis script first')
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

windowed = load_csv('windowed_correlation.csv')
spikes = load_csv('spike_summary.csv')
case_ctrl = load_csv('case_control_results.csv')
spike_det = load_csv('spike_detection.csv')

print(f'Windowed: {len(windowed)} windows')
print(f'Spike summary: {len(spikes)} experiments')
print(f'Case/control: {len(case_ctrl)} experiments')
print(f'Spike detection: {len(spike_det)} experiments')

## 2. Spike Window Distribution — Interactive Selection

In [ ]:
# ── Change TARGET_EXP to explore different experiments ──
TARGET_EXP = 'e7-full-stress'

exp_windows = [w for w in windowed if w['experiment'] == TARGET_EXP]
if exp_windows:
    times = [float(w['window_start_s']) for w in exp_windows]
    p99s = [float(w['app_p99_ms']) for w in exp_windows]
    classes = [w['spike_class'] for w in exp_windows]
    
    colors = ['red' if c == 'spike' else 'green' if c == 'calm' else 'orange' for c in classes]
    
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.scatter(times, p99s, c=colors, s=3, alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Window p99 (ms)')
    n_spike = sum(1 for c in classes if c == 'spike')
    ax.set_title(f'{TARGET_EXP}: {n_spike}/{len(classes)} spike windows ({n_spike/len(classes)*100:.1f}%)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f'No window data for {TARGET_EXP}')

## 3. Case/Control Summary Table

In [ ]:
if case_ctrl:
    print(f'{"Experiment":<30} {"n_total":>8} {"n_case":>8} {"n_ctrl":>8} {"Ratio":>8} {"Cliff δ":>8} {"p-val":>8}')
    print('─' * 85)
    for r in case_ctrl:
        print(f"{r['experiment']:<30} {r['n_total']:>8} {r['n_case']:>8} {r['n_control']:>8} "
              f"{r.get('case_control_ratio',''):>8} {r.get('window_cliffs_delta',''):>8} "
              f"{r.get('window_p_value',''):>8}")
else:
    print('No case/control data — run case_control_analysis.py first')

## 4. Spike Detection Summary

In [ ]:
if spike_det:
    exps = [r['experiment'] for r in spike_det]
    spike_pcts = [float(r['spike_pct']) for r in spike_det]
    n_bursts = [int(r['n_spike_bursts']) for r in spike_det]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    y = np.arange(len(exps))
    colors = ['#e74c3c' if p > 50 else '#f39c12' if p > 10 else '#2ecc71' for p in spike_pcts]
    ax1.barh(y, spike_pcts, color=colors)
    ax1.set_yticks(y)
    ax1.set_yticklabels(exps, fontsize=9)
    ax1.set_xlabel('Spike %')
    ax1.set_title('Spike Window Percentage')
    ax1.invert_yaxis()
    
    ax2.barh(y, n_bursts, color='#e67e22')
    ax2.set_yticks(y)
    ax2.set_yticklabels(exps, fontsize=9)
    ax2.set_xlabel('N Bursts')
    ax2.set_title('Spike Burst Count')
    ax2.invert_yaxis()
    
    plt.tight_layout()
    plt.show()
else:
    print('No spike detection data — run spike_detection.py first')

## 5. Custom Correlation Explorer

Pick any two fields from the windowed data and compute correlation.

In [ ]:
# ── Customize these ──
X_FIELD = 'throughput'
Y_FIELD = 'app_p99_ms'
FILTER_EXP = 'e7-full-stress'  # or None for all

filtered = [w for w in windowed if (FILTER_EXP is None or w['experiment'] == FILTER_EXP)]
if filtered:
    x = np.array([float(w[X_FIELD]) for w in filtered])
    y = np.array([float(w[Y_FIELD]) for w in filtered])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(x, y, s=5, alpha=0.4)
    ax.set_xlabel(X_FIELD)
    ax.set_ylabel(Y_FIELD)
    
    # Compute Pearson r
    r = np.corrcoef(x, y)[0, 1]
    title = f'{X_FIELD} vs {Y_FIELD}'
    if FILTER_EXP:
        title += f' ({FILTER_EXP})'
    ax.set_title(f'{title}\nPearson r = {r:.3f}, n = {len(x)}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No data matched filter')